## ▶ What you'll see when you run this
- A bar chart with **known** values, read back by Claude/Gemini as JSON, then scored against ground truth — e.g. `Claude MAE : 2.0`.

**Time:** ~10 min · **Cost:** free (cheapest model: Gemini Flash / Claude Haiku) · **Keys:** none to build the chart — add `ANTHROPIC_API_KEY` and/or `GEMINI_API_KEY` for the vision read-back (each skipped gracefully if missing).

# Week 15 · Notebook 2 — Reading Charts & Structured Extraction
**CSCI 250 · Fall 2026**

Generate a **bar chart with known values**, ask a vision model to read it back as **JSON**, then **check its accuracy against the ground truth**. This is the core skill behind the Final Project 'chart reader' / 'receipt reader' (the Multimodal track).

> Cells degrade gracefully if no API key is set.

## 0. Install + keys

In [ ]:
!pip -q install anthropic google-generativeai pillow matplotlib

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except Exception:
    pass
HAVE_CLAUDE = bool(os.environ.get('ANTHROPIC_API_KEY'))
HAVE_GEMINI = bool(os.environ.get('GEMINI_API_KEY'))
print('Claude:', HAVE_CLAUDE, '| Gemini:', HAVE_GEMINI)

## 1. Build a chart with KNOWN values (ground truth)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

TRUTH = {'North': 42, 'South': 27, 'East': 58, 'West': 19}
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(TRUTH.keys()), list(TRUTH.values()), color='#128C8C')
ax.set_title('Sales by Region (units)')
ax.set_ylabel('Units sold')
fig.savefig('chart.png', dpi=110, bbox_inches='tight')
plt.close(fig)
print('ground truth:', TRUTH)

In [ ]:
from PIL import Image
Image.open('chart.png')   # preview

## 2. Vision helpers (return raw text)

In [ ]:
import base64

def ask_claude(image_path, prompt, model='claude-sonnet-4-6'):
    if not HAVE_CLAUDE:
        return '[Claude skipped — no key]'
    import anthropic
    client = anthropic.Anthropic()
    b64 = base64.standard_b64encode(open(image_path, 'rb').read()).decode()
    msg = client.messages.create(
        model=model, max_tokens=500,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64',
                'media_type': 'image/png', 'data': b64}},
            {'type': 'text', 'text': prompt}]}])
    return msg.content[0].text

def ask_gemini(image_path, prompt, model='gemini-2.5-flash'):
    if not HAVE_GEMINI:
        return '[Gemini skipped — no key]'
    import google.generativeai as genai
    from PIL import Image
    genai.configure(api_key=os.environ['GEMINI_API_KEY'])
    m = genai.GenerativeModel(model)
    return m.generate_content([prompt, Image.open(image_path)]).text

## 3. Ask for the chart as JSON

In [ ]:
CHART_PROMPT = (
    'This is a bar chart. Return ONLY a JSON object mapping each bar label '
    'to its approximate numeric value, e.g. {"North": 40}. No prose, no '
    'code fences.')
claude_raw = ask_claude('chart.png', CHART_PROMPT)
gemini_raw = ask_gemini('chart.png', CHART_PROMPT)
print('CLAUDE raw:', claude_raw)
print('GEMINI raw:', gemini_raw)

## 4. Parse JSON robustly
Models sometimes wrap JSON in ```code fences``` or add a sentence. This helper pulls out the first `{...}` block and parses it.

In [ ]:
import json, re

def parse_json(text):
    if not text or text.startswith('['):
        return None
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

claude_json = parse_json(claude_raw)
gemini_json = parse_json(gemini_raw)
print('Claude parsed:', claude_json)
print('Gemini parsed:', gemini_json)

## 5. Score against ground truth
Mean absolute error (MAE) between the model's read and the true values.

In [ ]:
def score(pred, truth):
    if not pred:
        return None
    errs = []
    for label, true_val in truth.items():
        if label in pred:
            try:
                errs.append(abs(float(pred[label]) - true_val))
            except (TypeError, ValueError):
                pass
    return round(sum(errs) / len(errs), 2) if errs else None

print('TRUTH      :', TRUTH)
print('Claude MAE :', score(claude_json, TRUTH))
print('Gemini MAE :', score(gemini_json, TRUTH))

## 6. Reflect
- How close were the read-off values? Which model was more accurate?
- VLMs estimate values **from pixels** — good for 'which bar is biggest', risky for exact figures. When would that error matter in your project?
- **Extend it:** swap in a receipt photo and a JSON schema (`vendor, date, items[], total`) to prototype the Track D 'Receipt Reader'.

In [ ]:
# notes / your extension here:
